# 03 · Segundo aluno: BERT para classificação de tokens

Corre com **Runtime → Change runtime type → T4 GPU**.
Usa os mesmos `train/val/test.jsonl` do notebook 02 e compara-se com ele no mesmo teste de ouro.

## 1. Preparar o ambiente

In [ ]:
# Instalar dependências (awscli: o Colab não traz a CLI da AWS)
!pip install -q -U transformers accelerate onnx onnxruntime datasets boto3 pydantic awscli

import os
from google.colab import drive, userdata

# Secrets do Colab (ícone da chave, à esquerda): AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, DATA_BUCKET
for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "DATA_BUCKET"):
    os.environ[k] = userdata.get(k)
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

drive.mount("/content/drive")  # tudo o que demora vai para o Drive: sobrevive a desconexões
DRIVE = "/content/drive/MyDrive/wiki-bio-distill"
os.makedirs(DRIVE, exist_ok=True)

# Repositório: público, ou privado com um secret GITHUB_TOKEN
try:
    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/pinthoz/wiki-bio-distill.git"
except Exception:
    url = "https://github.com/pinthoz/wiki-bio-distill.git"
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
%cd /content/repo
!git pull -q
!mkdir -p data preds
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!for f in raw train val test; do aws s3 cp s3://$DATA_BUCKET/v1/$f.jsonl data/; done
# previsões do notebook 02 (se já existirem), para comparar tudo na mesma tabela
!aws s3 cp s3://$DATA_BUCKET/v1/preds/ preds/ --recursive || true

BERT = "neuralmind/bert-base-portuguese-cased"   # alternativa PT-PT: "PORTULAN/albertina-100m-portuguese-ptpt-encoder"
RUN_DIR = f"{DRIVE}/bert-v1"
MAX_LEN = 384

## 2. JSON do professor → spans no texto

Mostra quantos exemplos ficaram totalmente alinhados. Os que não alinham ficam fora do treino.

In [ ]:
!python bert_data.py
!head -c 600 data/bert_train.jsonl

## 3. Tokenizar e criar as etiquetas BIO

In [ ]:
import json, time, torch
from datasets import load_dataset
from transformers import (AutoModelForTokenClassification, AutoTokenizer,
                          DataCollatorForTokenClassification, Trainer, TrainingArguments)
from bert_data import LABELS, LABEL2ID, spans_from_bio, spans_to_json

tokenizer = AutoTokenizer.from_pretrained(BERT)

def encode(ex):
    enc = tokenizer(ex["text"], truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
    labels = []
    for s, e in enc["offset_mapping"]:
        if s == e:                                   # [CLS], [SEP]: ignorados na loss
            labels.append(-100)
            continue
        tag = "O"
        for sp in ex["spans"]:
            if s >= sp["start"] and e <= sp["end"]:  # token dentro da span
                tag = ("B-" if s == sp["start"] else "I-") + sp["label"]
                break
        labels.append(LABEL2ID[tag])
    enc["labels"] = labels
    enc.pop("offset_mapping")
    return enc

data = load_dataset("json", data_files={"train": "data/bert_train.jsonl", "validation": "data/bert_val.jsonl"})
tok_data = data.map(encode, remove_columns=data["train"].column_names)

# confirmar num exemplo que as etiquetas caem nas palavras certas
ex = tok_data["train"][0]
for t, l in zip(tokenizer.convert_ids_to_tokens(ex["input_ids"])[:40], ex["labels"][:40]):
    print(f"{t:>15}  {LABELS[l] if l >= 0 else '-'}")

## 4. Treinar (poucos minutos numa T4)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    BERT, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)

args = TrainingArguments(
    output_dir=RUN_DIR, num_train_epochs=5, per_device_train_batch_size=16,
    learning_rate=5e-5, weight_decay=0.01, warmup_steps=50,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    fp16=True, report_to="none", logging_steps=20,
)
trainer = Trainer(model=model, args=args, train_dataset=tok_data["train"],
                  eval_dataset=tok_data["validation"],
                  data_collator=DataCollatorForTokenClassification(tokenizer))
trainer.train()
trainer.save_model(f"{RUN_DIR}/best")
tokenizer.save_pretrained(f"{RUN_DIR}/best")

## 5. Prever o teste de ouro e comparar com os outros modelos

In [ ]:
lemma_map = json.load(open("data/lemma_map.json", encoding="utf-8"))
test = [json.loads(l) for l in open("data/test.jsonl", encoding="utf-8")]
model.eval()
with open("preds/bert.jsonl", "w", encoding="utf-8") as f, torch.no_grad():
    for t in test:
        t0 = time.perf_counter()
        enc = tokenizer(t["text"], truncation=True, max_length=MAX_LEN,
                        return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping")[0].tolist()
        logits = model(**{k: v.to(model.device) for k, v in enc.items()}).logits[0]
        tags = [LABELS[i] for i in logits.argmax(-1).tolist()]
        pred = spans_to_json(spans_from_bio(t["text"], offsets, tags), lemma_map)
        f.write(json.dumps({"id": t["id"], "raw": json.dumps(pred, ensure_ascii=False),
                            "ms": round((time.perf_counter() - t0) * 1000)}, ensure_ascii=False) + "\n")

import glob
others = [p for p in ("preds/base.jsonl", "preds/student.jsonl", "preds/student-q4.jsonl") if glob.glob(p)]
others_str = " ".join(others)
!python evaluate.py {others_str} preds/bert.jsonl

## 6. Análise de erros do BERT

Repara se os erros vêm do modelo (span errada) ou do pós-processamento (data mal normalizada, lema errado).

In [ ]:
from common import FIELDS, as_set, parse_json
preds = {p["id"]: parse_json(p["raw"]) for p in map(json.loads, open("preds/bert.jsonl"))}
shown = 0
for t in test:
    p = preds[t["id"]] or {}
    wrong = [f for f in FIELDS if as_set(p.get(f)) != as_set(t["gold"].get(f))]
    if wrong and shown < 10:
        shown += 1
        print(t["text"][:200], "…")
        for f in wrong:
            print(f"  {f}: ouro={t['gold'].get(f)!r}  bert={p.get(f)!r}")
        print()

## 7. Exportar para ONNX (igual ao bias-api) e guardar no S3

In [ ]:
from pathlib import Path
import onnxruntime as ort, numpy as np

OUT = Path("/content/bert-onnx"); OUT.mkdir(exist_ok=True)
cpu_model = AutoModelForTokenClassification.from_pretrained(f"{RUN_DIR}/best").eval()
sample = tokenizer("Uma frase de exemplo.", return_tensors="pt")
names = list(sample.keys())
dynamic = {n: {0: "batch", 1: "seq"} for n in names}
dynamic["logits"] = {0: "batch", 1: "seq"}
torch.onnx.export(cpu_model, args=(), kwargs=dict(sample), f=str(OUT / "model.onnx"),
                  input_names=names, output_names=["logits"], dynamic_axes=dynamic,
                  opset_version=17, dynamo=False)
tokenizer.save_pretrained(OUT)
!cp data/lemma_map.json {OUT}/

# validação: PyTorch e ONNX têm de dar os mesmos logits
with torch.no_grad():
    ref = cpu_model(**sample).logits.numpy()
out = ort.InferenceSession(str(OUT / "model.onnx")).run(None, {k: v.numpy() for k, v in sample.items()})[0]
print("diferença máxima:", abs(ref - out).max())
!ls -lh {OUT}

In [ ]:
!aws s3 cp --recursive /content/bert-onnx s3://$DATA_BUCKET/v1/models/bert-token/
!aws s3 cp preds/bert.jsonl s3://$DATA_BUCKET/v1/preds/
!aws s3 cp results.json s3://$DATA_BUCKET/v1/results-with-bert.json